# 02 · *Explainable AI* Berbasis SHAP (Tahap C11–C15)
## Luaran artikel 2026: interpretasi prediksi tingkat keparahan dermatitis atopik pada bayi

Notebook ini menjelaskan keputusan CNN keparahan pada ROI hasil deteksi YOLO. SHAP **hanya menerangkan CNN pada ROI**; proses deteksi YOLO berada di luar cakupan penjelasan, dan batasan ini juga dinyatakan di naskah.

| Tahap | Pekerjaan | Keluaran |
|---|---|---|
| C11 | SHAP *Partition explainer* (masker *blur*) dan *GradientExplainer* (*expected gradients*, dengan *background* representatif) | `shap_values/*.npz`, `sample_manifest.csv` |
| C12 | *heatmap* per kelas Ringan, Sedang, Berat; contoh benar dan salah; konsentrasi atribusi pada lesi | `heatmaps/*.png`, gambar per kelas |
| C13 | *faithfulness* (*deletion/insertion*), stabilitas, dan *sanity check* (randomisasi model) | `faithfulness.csv`, `stability.csv`, `sanity_check.csv` |
| C14 | paket validasi dokter anonim dan analisis kesepakatan (bila penilaian nyata sudah ada) | `expert_validation/`, `agreement.csv` |
| C15 | prototipe unggah → YOLO → ROI → CNN → SHAP dan uji kasus | `prototype/`, `test_report.csv` |

Rujukan metode utama: Lundberg & Lee (2017) untuk SHAP; Petsiuk et al. (2018) untuk *deletion/insertion*; Alvarez-Melis & Jaakkola (2018) dan Yeh et al. (2019) untuk stabilitas/sensitivitas penjelasan; Adebayo et al. (2018) untuk *sanity check* randomisasi model; Selvaraju et al. (2017) untuk Grad-CAM sebagai pembanding.

## 0. Konfigurasi

- `MODEL_SET_XAI`: model yang dijelaskan (bawaan `retrain`, model dengan *test set* sah).
- `N_BENAR_PER_KELAS`, `N_SALAH_PER_KELAS`: jumlah ROI *test* per kelas sebenarnya yang dijelaskan (prediksi benar dan salah).
- `SHAP_MAX_EVALS`: anggaran evaluasi model per keluaran untuk *Partition explainer*; makin besar makin halus petanya.
- `PATCH`: ukuran *patch* (piksel) untuk peringkat *deletion/insertion* dan perbandingan peta (224/8 = 28 × 28 *patch*).
- `STABILITAS_GANGGUAN`: gangguan kecil yang secara klinis tidak mengubah lesi, untuk menguji stabilitas peta.
- `LAUNCH_GRADIO = True` menampilkan antarmuka prototipe di Colab.

In [ ]:
# ======================= CONFIG =======================
NAMA_PROYEK          = "YOLO_DERMA_PFR"
MODE_UJI_CEPAT       = False
MODEL_SET_XAI        = "retrain"

N_BENAR_PER_KELAS    = 20
N_SALAH_PER_KELAS    = 10
SHAP_MAX_EVALS       = 1000
SHAP_BATCH           = 50
SHAP_MASKER          = "blur(16,16)"
JALANKAN_GRADIENT_SHAP = True
N_BACKGROUND         = 102        # dibagi rata ke 3 kelas (ROI referensi dari split train)
GRAD_NSAMPLES        = 200

PATCH                = 8
DEL_INS_STEPS        = 50
N_URUTAN_ACAK        = 5
STABILITAS_GANGGUAN  = [("gaussian_noise", 2.0), ("brightness", 1.05), ("brightness", 0.95), ("jpeg", 95)]
TOP_FRAKSI           = 0.10
SANITY_N             = 20
SANITY_MAX_EVALS     = 500

N_KASUS_DOKTER       = 60
N_ROI_SHAP_PROTOTIPE = 3
SHAP_MAX_EVALS_PROTO = 300
LAUNCH_GRADIO        = False
B_BOOT               = 1000

## 1. Lingkungan, data, dan model

Sel ini membaca konfigurasi dari notebook 00, memulihkan data dari *cache* Drive, dan memuat model yang dijelaskan beserta skala *preprocessing* dan urutan kelasnya dari `model_config.json`.

In [ ]:
import os, sys, subprocess, importlib, time, json, shutil, glob, re, math, warnings, tarfile, datetime, io
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
IN_COLAB = "google.colab" in sys.modules
for mod, pkg in [("ultralytics", "ultralytics"), ("imagehash", "imagehash"), ("shap", "shap"), ("krippendorff", "krippendorff"),
                 ("openpyxl", "openpyxl")] + ([("gradio", "gradio")] if LAUNCH_GRADIO else []):
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import torch, torchvision, ultralytics  # noqa: F401

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_BASE, WORK = "/content/drive/MyDrive", "/content/work"
else:
    DRIVE_BASE, WORK = os.path.abspath("drive_lokal"), os.path.abspath("work_lokal")
if MODE_UJI_CEPAT:
    NAMA_PROYEK += "_UJI_CEPAT"
    N_BENAR_PER_KELAS, N_SALAH_PER_KELAS, SHAP_MAX_EVALS, GRAD_NSAMPLES = 2, 1, 100, 10
    DEL_INS_STEPS, N_URUTAN_ACAK, SANITY_N, SANITY_MAX_EVALS, N_KASUS_DOKTER = 10, 2, 3, 100, 6
    N_BACKGROUND, SHAP_MAX_EVALS_PROTO, B_BOOT = 12, 100, 100
ROOT = os.path.join(DRIVE_BASE, NAMA_PROYEK)
P = {"kode": f"{ROOT}/00_kode", "data": f"{ROOT}/01_data_cache", "A": f"{ROOT}/A_PERSIAPAN", "A_tab": f"{ROOT}/A_PERSIAPAN/tables",
     "C": f"{ROOT}/PFR_2026_XAI_SHAP"}
P.update({k: f"{P['C']}/{v}" for k, v in {"C_tab": "tables", "C_fig": "figures", "C_shap": "shap_values", "C_heat": "heatmaps",
                                           "C_exp": "expert_validation", "C_proto": "prototype", "C_log": "logs"}.items()})
for v in P.values():
    os.makedirs(v, exist_ok=True)
os.makedirs(WORK, exist_ok=True)
cfg_path = os.path.join(P["A"], "config_run.json")
if not os.path.exists(cfg_path):
    raise FileNotFoundError("config_run.json tidak ditemukan. Jalankan notebook 00 terlebih dahulu.")
sys.path.insert(0, P["kode"])
import dermakit as dk
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import shap
CFG = dk.core.load_json(cfg_path)
MC = dk.core.load_json(os.path.join(P["A_tab"], "model_config.json"))
SEED = CFG["SEED"]; dk.core.set_seed(SEED); dk.viz.setup_style()
STATUS_LABEL_MAP = CFG["STATUS_LABEL_MAP"]
CONF_THR, IOU_VALID = MC["inferensi"]["conf_thr"], MC["inferensi"]["iou_valid"]
NAMA_KELAS = dk.core.class_names(CFG.get("BAHASA_GAMBAR", "en")); NAMA_KELAS_ID = dk.core.class_names("id")
LOG = dk.core.RunLog(os.path.join(P["C_log"], "run_log_02.txt"))
REPO_DIR = CFG.get("REPO_LOKAL") or os.path.join(WORK, "YOLO-DERMA")
if not os.path.exists(os.path.join(REPO_DIR, "train")):
    with tarfile.open(os.path.join(P["data"], "yolo_derma_repo_cache.tar")) as t:
        t.extractall(WORK)
split_df = pd.read_csv(os.path.join(P["A_tab"], "locked_data_split.csv"))
ver = pd.read_csv(os.path.join(P["A_tab"], "verified_labels.csv"))

tf = dk.models.get_tf()
import keras
m_cfg = MC[MODEL_SET_XAI]
if MODEL_SET_XAI == "retrain":
    cnn_model = keras.models.load_model(m_cfg["cnn_path"], compile=False)
else:
    cnn_model = dk.models.load_keras_any(m_cfg["cnn_path"])[0]
YR = dk.models.YoloRunner(m_cfg["yolo_path"], imgsz=MC["inferensi"]["yolo_imgsz"], nms_iou=MC["inferensi"]["nms_iou"],
                          max_det=MC["inferensi"]["max_det"], keep_classes=m_cfg.get("yolo_keep_classes"), name=MODEL_SET_XAI)
CR = dk.models.CnnRunner(cnn_model, scale=m_cfg["cnn_scale"], class_order=m_cfg["cnn_class_order"], pad=m_cfg["cnn_pad"], name=MODEL_SET_XAI)
S = CR.input_size
F_PROB = CR.predict_fn()
print("Model XAI:", MODEL_SET_XAI, "| input CNN:", S, "| skala:", CR.scale, "| urutan kelas:", CR.class_order, "| label map:", STATUS_LABEL_MAP)

## C11. Perhitungan SHAP untuk probabilitas kelas CNN pada ROI

### C11.1 Pemilihan sampel evaluasi

Sampel diambil dari hasil *baseline* bersih notebook 00 pada *test set* terkunci: lesi dengan ROI prediksi YOLO yang valid (IoU ≥ 0,5 terhadap lesi referensi), sehingga kelas sebenarnya diketahui dan ROI yang dijelaskan adalah ROI yang benar-benar dipakai *pipeline*. Untuk setiap kelas sebenarnya dipilih `N_BENAR_PER_KELAS` prediksi benar dan `N_SALAH_PER_KELAS` prediksi salah secara acak (*seed* tetap), dengan paling banyak satu ROI per citra agar sampel tidak didominasi satu citra. Daftar sampel dikunci di `sample_manifest.csv`.

In [ ]:
base = pd.read_csv(os.path.join(P["A_tab"], f"baseline_predictions_roi_{MODEL_SET_XAI}.csv"))
paths = dict(zip(split_df.sample_id, split_df.file_rel.map(lambda p: os.path.join(REPO_DIR, p))))
if "pb_x1" not in base.columns:
    print("Kolom kotak prediksi tidak ada (notebook 00 versi lama); kotak dipulihkan dengan inferensi ulang.")
    rows = []
    for sid, g in base[base.has_overlap.astype(bool)].groupby("sample_id"):
        im = dk.data.read_rgb(paths[sid]); pr = YR.predict([im], conf=0.001)[0][0]; pr = pr[pr[:, 4] >= CONF_THR]
        gt = ver[(ver.sample_id == sid) & ver.keep & (ver.sev >= 0)].sort_values("box_idx")[["x1", "y1", "x2", "y2"]].to_numpy()
        bp, _ = dk.geometry.hungarian_match(gt, pr[:, :4])
        for r in g.itertuples():
            j = bp[r.roi_idx]
            rows.append({"sample_id": sid, "roi_idx": r.roi_idx, "pb_x1": pr[j, 0], "pb_y1": pr[j, 1], "pb_x2": pr[j, 2], "pb_y2": pr[j, 3],
                         "gt_x1": gt[r.roi_idx, 0], "gt_y1": gt[r.roi_idx, 1], "gt_x2": gt[r.roi_idx, 2], "gt_y2": gt[r.roi_idx, 3]})
    base = base.merge(pd.DataFrame(rows), on=["sample_id", "roi_idx"], how="left")

cand = base[base.valid_roi.astype(bool)].copy()
cand["correct"] = cand.any_pred == cand.true
rng = np.random.default_rng(SEED)
cand = cand.sample(frac=1.0, random_state=SEED)
pilih = []
for k in range(3):
    for benar, n in [(True, N_BENAR_PER_KELAS), (False, N_SALAH_PER_KELAS)]:
        q = cand[(cand.true == k) & (cand.correct == benar)].drop_duplicates("sample_id").head(n)
        pilih.append(q)
man = pd.concat(pilih, ignore_index=True)
man.insert(0, "xai_id", [f"X{i:03d}" for i in range(len(man))])
man["path"] = man.sample_id.map(paths)
man["pred"] = man.any_pred.astype(int)
man[["xai_id", "sample_id", "group", "roi_idx", "true", "pred", "correct", "any_p0", "any_p1", "any_p2", "iou", "det_conf",
     "gt_x1", "gt_y1", "gt_x2", "gt_y2", "pb_x1", "pb_y1", "pb_x2", "pb_y2", "path"]].to_csv(os.path.join(P["C_tab"], "sample_manifest.csv"), index=False)
print(pd.crosstab(man.true.map(dict(enumerate(NAMA_KELAS_ID))), man.correct.map({True: "benar", False: "salah"})))

IMGS = {sid: dk.data.read_rgb(paths[sid]) for sid in man.sample_id.unique()}
PB = man[["pb_x1", "pb_y1", "pb_x2", "pb_y2"]].to_numpy(float)
X = np.concatenate([CR.crops(IMGS[r.sample_id], PB[i:i + 1]) for i, r in enumerate(man.itertuples())])
prob0 = F_PROB(X)
selisih = np.abs(prob0 - man[["any_p0", "any_p1", "any_p2"]].to_numpy(float)).max()
print("Crop ROI:", X.shape, f"| selisih probabilitas maksimum terhadap baseline notebook 00 = {selisih:.2e}")
if selisih > 1e-2:
    print("PERINGATAN: probabilitas berbeda dari baseline; periksa apakah model atau data berubah setelah notebook 00.")

### C11.2 SHAP *Partition explainer* (masker *blur*)

*Partition explainer* menghitung nilai Owen (pendekatan nilai Shapley pada hierarki partisi citra). Bagian citra yang "disembunyikan" diganti versi kabur (*blur* 16×16) dari ROI yang sama, sehingga referensinya tetap berada pada distribusi warna kulit yang wajar. Atribusi dihitung untuk ketiga kelas (Ringan, Sedang, Berat) dan dijumlahkan atas kanal warna menjadi satu peta per kelas. Sifat aditif diperiksa: nilai dasar + jumlah atribusi = probabilitas model.

In [ ]:
def shap_partition(Xin, outputs=(0, 1, 2), max_evals=SHAP_MAX_EVALS, fprob=F_PROB):
    ex = shap.Explainer(fprob, shap.maskers.Image(SHAP_MASKER, Xin.shape[1:]), output_names=NAMA_KELAS)
    sv = ex(Xin, max_evals=max_evals, batch_size=SHAP_BATCH, outputs=list(outputs), silent=True)
    return sv.values.sum(axis=3), sv.base_values

t0 = time.time()
SV_PART, BASE_PART = shap_partition(X)
dt_part = time.time() - t0
aditif = np.abs(BASE_PART + SV_PART.sum(axis=(1, 2)) - prob0).max()
np.savez_compressed(os.path.join(P["C_shap"], "shap_values_partition.npz"), values=SV_PART.astype(np.float16),
                    base_values=BASE_PART, xai_id=man.xai_id.values, kelas=np.array(NAMA_KELAS))
print(f"Partition SHAP: {SV_PART.shape}, {dt_part:.0f} detik, galat aditivitas maksimum {aditif:.2e}")

### C11.3 *GradientExplainer* (*expected gradients*) dengan *background* representatif

Sebagai penjelasan kedua yang memakai himpunan *background* eksplisit, *expected gradients* (Erion et al., 2021) mengintegrasikan gradien sepanjang jalur dari sampel *background* ke citra. *Background* = `N_BACKGROUND` ROI referensi dari *split train*, dibagi rata per kelas dan diambil dari citra berbeda, sehingga mewakili distribusi ROI latih. Masukan diskalakan sesuai `CnnRunner.scale`, dan keluaran model diurutkan ulang ke urutan Ringan, Sedang, Berat.

In [ ]:
SV_GRAD = None
if JALANKAN_GRADIENT_SHAP:
    rois_tr = ver[ver.keep & (ver.sev >= 0)].merge(split_df[split_df.split == "train"][["sample_id", "file_rel"]], on="sample_id")
    rois_tr = rois_tr.sample(frac=1.0, random_state=SEED).drop_duplicates("sample_id")
    bg = pd.concat([rois_tr[rois_tr.sev == k].head(N_BACKGROUND // 3) for k in range(3)])
    XB = np.concatenate([CR.crops(dk.data.read_rgb(os.path.join(REPO_DIR, r.file_rel)), np.array([[r.x1, r.y1, r.x2, r.y2]])) for r in bg.itertuples()])
    bg[["sample_id", "box_idx", "sev"]].to_csv(os.path.join(P["C_tab"], "background_manifest.csv"), index=False)
    try:
        t0 = time.time()
        ge = shap.GradientExplainer(CR.model, dk.models.scale_input(XB, CR.scale))
        raw = ge.shap_values(dk.models.scale_input(X, CR.scale), nsamples=GRAD_NSAMPLES, rseed=SEED)
        raw = np.asarray(raw)
        if raw.ndim == 5 and raw.shape[0] != len(X):
            raw = np.moveaxis(raw, 0, -1)
        raw = raw.sum(axis=3)
        SV_GRAD = np.zeros(raw.shape[:3] + (3,), np.float32)
        for i, k in enumerate(CR.class_order):
            if 0 <= k < 3:
                SV_GRAD[..., k] += raw[..., i]
        np.savez_compressed(os.path.join(P["C_shap"], "shap_values_gradient.npz"), values=SV_GRAD.astype(np.float16), xai_id=man.xai_id.values)
        print(f"GradientExplainer: {SV_GRAD.shape}, {time.time() - t0:.0f} detik, background {len(XB)} ROI")
    except Exception as e:
        print("GradientExplainer gagal dan dilewati:", type(e).__name__, str(e)[:200])
        SV_GRAD = None

def gradcam(Xin, kelas):
    try:
        layer = CR.model.get_layer("top_activation")
    except Exception:
        return None
    gm = keras.Model(CR.model.inputs, [layer.output, CR.model.output])
    out = np.zeros(Xin.shape[:3], np.float32)
    for i in range(len(Xin)):
        xi = tf.convert_to_tensor(dk.models.scale_input(Xin[i:i + 1], CR.scale))
        with tf.GradientTape() as tape:
            a, p = gm(xi, training=False)
            s = p[:, CR.class_order.index(int(kelas[i]))]
        g = tape.gradient(s, a)
        w = tf.reduce_mean(g, axis=(1, 2), keepdims=True)
        cam = tf.nn.relu(tf.reduce_sum(w * a, axis=-1))[0].numpy()
        out[i] = tf.image.resize(cam[..., None], Xin.shape[1:3]).numpy()[..., 0]
    return out
GC = gradcam(X, man.pred.values)
print("Grad-CAM:", "tersedia" if GC is not None else "tidak tersedia untuk arsitektur ini")
LOG("C11 selesai.")

## C12. Visualisasi *heatmap* SHAP per kelas

### C12.1 Panel per sampel

Setiap sampel disimpan sebagai satu panel konsisten: (a) citra asli dengan kotak lesi referensi (putus-putus) dan kotak YOLO (garis tegas), (b) ROI masukan CNN, (c–e) peta SHAP untuk Ringan, Sedang, dan Berat dengan skala warna bersama (merah = menaikkan probabilitas kelas, biru = menurunkan), dan (f) tiga probabilitas CNN. Berkas tersimpan di `heatmaps/<kelas sebenarnya>/<xai_id>_<benar|salah>.png`.

In [ ]:
def panel_sampel(i, simpan=True, tampil=False):
    r = man.iloc[i]
    img = IMGS[r.sample_id]
    fig, ax = plt.subplots(1, 6, figsize=(22, 4.3), gridspec_kw={"width_ratios": [1.25, 1, 1, 1, 1, 0.9]})
    ax[0].imshow(img)
    ax[0].add_patch(plt.Rectangle((r.gt_x1, r.gt_y1), r.gt_x2 - r.gt_x1, r.gt_y2 - r.gt_y1, fill=False, ec="white", ls="--", lw=2))
    ax[0].add_patch(plt.Rectangle((r.pb_x1, r.pb_y1), r.pb_x2 - r.pb_x1, r.pb_y2 - r.pb_y1, fill=False, ec="#FFD400", lw=2))
    ax[0].set_title("(a) Image: ref. (dashed), YOLO (solid)", fontsize=12)
    ax[1].imshow(X[i].astype(np.uint8)); ax[1].set_title("(b) ROI input", fontsize=12)
    vmax = np.nanpercentile(np.abs(SV_PART[i]), 99.5) + 1e-12
    for k in range(3):
        ax[2 + k].imshow(dk.viz.overlay_heatmap(X[i].astype(np.uint8), SV_PART[i, ..., k], vmax=vmax))
        ax[2 + k].set_title(f"({'cde'[k]}) SHAP: {NAMA_KELAS[k]}", fontsize=12)
    for a in ax[:5]:
        a.axis("off")
    p = [r.any_p0, r.any_p1, r.any_p2]
    bars = ax[5].barh(NAMA_KELAS, p, color=dk.viz.CLASS_COLORS)
    ax[5].bar_label(bars, labels=[f"{v:.2f}" for v in p], fontsize=12, padding=2)
    ax[5].set_xlim(0, 1.2); ax[5].invert_yaxis(); ax[5].set_title("(f) CNN probability", fontsize=12)
    fig.suptitle(f"{r.xai_id} | true: {NAMA_KELAS[r.true]} | predicted: {NAMA_KELAS[r.pred]} | "
                 f"{'correct' if r.correct else 'incorrect'} | IoU {r.iou:.2f}", fontsize=14)
    if simpan:
        d = os.path.join(P["C_heat"], NAMA_KELAS[r.true]); os.makedirs(d, exist_ok=True)
        dk.viz.save_fig(fig, os.path.join(d, f"{r.xai_id}_{'benar' if r.correct else 'salah'}"))
    if tampil:
        plt.show()
    plt.close(fig)

for i in range(len(man)):
    panel_sampel(i, tampil=(i in set(man.groupby(["true", "correct"]).head(1).index)))
print("Panel tersimpan:", len(glob.glob(os.path.join(P["C_heat"], "*", "*.png"))))

### C12.2 Gambar ringkas per kelas dan peta rata-rata

Gambar ringkas (untuk naskah) memuat baris per kelas sebenarnya: tiga contoh benar dan dua contoh salah, masing-masing ROI dan peta SHAP untuk kelas yang diprediksi. Peta rata-rata |SHAP| per kelas menunjukkan di mana atribusi terkonsentrasi secara umum.

Konsentrasi atribusi pada lesi diukur dengan **rasio konsentrasi**: proporsi atribusi positif (kelas prediksi) yang jatuh di dalam kotak YOLO tanpa padding, dibagi proporsi luas kotak tersebut di dalam ROI. Nilai 1 berarti sebaran merata; nilai > 1 berarti atribusi terpusat pada area lesi. 95% CI dihitung dengan *cluster bootstrap* per kelompok sumber.

In [ ]:
fig, ax = plt.subplots(3, 10, figsize=(22, 7.6))
for k in range(3):
    idx = list(man[(man.true == k) & man.correct].index[:3]) + list(man[(man.true == k) & ~man.correct].index[:2])
    for c in range(5):
        a1, a2 = ax[k, 2 * c], ax[k, 2 * c + 1]
        a1.axis("off"); a2.axis("off")
        if c >= len(idx):
            continue
        i = idx[c]; r = man.loc[i]
        a1.imshow(X[i].astype(np.uint8))
        a2.imshow(dk.viz.overlay_heatmap(X[i].astype(np.uint8), SV_PART[i, ..., r.pred]))
        a1.set_title(f"{'✓' if r.correct else '✗'} pred {NAMA_KELAS[r.pred]}", fontsize=12)
    ax[k, 0].text(-0.15, 0.5, f"True: {NAMA_KELAS[k]}", transform=ax[k, 0].transAxes, rotation=90, va="center", ha="right", fontsize=14)
dk.viz.show_and_save(fig, os.path.join(P["C_fig"], "Gambar_SHAP_per_kelas"))

fig, ax = plt.subplots(1, 3, figsize=(13, 4.4))
for k in range(3):
    q = man.index[man.true == k]
    mmap = np.abs(SV_PART[q][np.arange(len(q)), ..., man.loc[q, "pred"].values]).mean(axis=0) if len(q) else np.zeros((S, S))
    im = ax[k].imshow(mmap, cmap="magma"); ax[k].axis("off"); ax[k].set_title(f"Mean |SHAP|: true {NAMA_KELAS[k]}")
    plt.colorbar(im, ax=ax[k], fraction=0.046)
dk.viz.show_and_save(fig, os.path.join(P["C_fig"], "Gambar_SHAP_rerata_per_kelas"))

def kotak_dalam(r):
    H, W = IMGS[r.sample_id].shape[:2]
    pbx = dk.geometry.pad_boxes(np.array([[r.pb_x1, r.pb_y1, r.pb_x2, r.pb_y2]]), CR.pad, W, H)[0]
    sx, sy = (S - 1) / max(pbx[2] - pbx[0], 1e-6), (S - 1) / max(pbx[3] - pbx[1], 1e-6)
    x1, x2 = int(round((r.pb_x1 - pbx[0]) * sx)), int(round((r.pb_x2 - pbx[0]) * sx))
    y1, y2 = int(round((r.pb_y1 - pbx[1]) * sy)), int(round((r.pb_y2 - pbx[1]) * sy))
    return max(0, x1), max(0, y1), min(S, x2 + 1), min(S, y2 + 1)
konsen = []
for i, r in man.iterrows():
    a = np.clip(SV_PART[i, ..., r.pred], 0, None)
    x1, y1, x2, y2 = kotak_dalam(r)
    luas = (x2 - x1) * (y2 - y1) / (S * S)
    dalam = a[y1:y2, x1:x2].sum() / max(a.sum(), 1e-12)
    konsen.append({"xai_id": r.xai_id, "group": r.group, "true": r.true, "correct": r.correct, "fraksi_luas_kotak": luas,
                   "fraksi_atribusi_positif_dalam_kotak": dalam, "rasio_konsentrasi": dalam / max(luas, 1e-6)})
konsen = pd.DataFrame(konsen)
konsen.to_csv(os.path.join(P["C_tab"], "konsentrasi_atribusi.csv"), index=False)
rows = []
for nm, q in [("Semua", konsen)] + [(NAMA_KELAS_ID[k], konsen[konsen.true == k]) for k in range(3)] + [("Benar", konsen[konsen.correct]), ("Salah", konsen[~konsen.correct])]:
    if len(q) == 0:
        continue
    cb = dk.stats.ClusterBootstrap(B_BOOT, SEED).register("q", q.group.astype(str).values)
    bs = [q.rasio_konsentrasi.to_numpy()[d["q"]].mean() for d in cb.draws()]
    lo, hi = dk.stats.percentile_ci(bs)
    rows.append({"kelompok": nm, "n": len(q), "rasio_konsentrasi_rerata": q.rasio_konsentrasi.mean(), "ci_lo": lo, "ci_hi": hi,
                 "fraksi_atribusi_dalam_kotak_rerata": q.fraksi_atribusi_positif_dalam_kotak.mean()})
konsen_sum = pd.DataFrame(rows)
konsen_sum.to_csv(os.path.join(P["C_tab"], "konsentrasi_atribusi_ringkas.csv"), index=False)
display(konsen_sum.round(3))
fmt, fci = dk.teks.fmt, dk.teks.fmt_ci
for r in konsen_sum.itertuples():
    print(f"{r.kelompok} (n = {r.n}): rasio konsentrasi {fmt(r.rasio_konsentrasi_rerata, 2)} (95% CI {fci(r.ci_lo, r.ci_hi, 2)}); "
          f"{dk.teks.pct(r.fraksi_atribusi_dalam_kotak_rerata)} atribusi positif berada di dalam kotak lesi. "
          + ("Atribusi terpusat pada area lesi (CI di atas 1)." if r.ci_lo > 1 else "Atribusi tersebar ke luar kotak (CI di bawah 1)." if r.ci_hi < 1 else "CI mencakup 1."))
LOG("C12 selesai.")

## C13. Uji mutu penjelasan

### C13.1 *Faithfulness*: *deletion* dan *insertion*

Protokol (Petsiuk et al., 2018) pada tingkat *patch* `PATCH`×`PATCH` piksel:

- *Patch* diurutkan menurut jumlah atribusi untuk kelas yang diprediksi (menurun).
- ***Deletion***: mulai dari ROI asli, *patch* teratas berturut-turut diganti versi *blur* (sama dengan masker SHAP) dalam `DEL_INS_STEPS` langkah; dicatat probabilitas kelas prediksi. AUC kecil = penjelasan menunjuk area yang benar-benar dipakai model.
- ***Insertion***: mulai dari ROI *blur*, *patch* teratas berturut-turut dikembalikan; AUC besar = baik.

Metode yang dibandingkan: SHAP *Partition*, SHAP *Gradient* (bila tersedia), Grad-CAM (bila tersedia), dan `N_URUTAN_ACAK` urutan acak sebagai *baseline*. Perbandingan berpasangan per sampel memakai uji *Wilcoxon signed-rank* dengan koreksi Holm. Penggantian dengan *blur* tetap menggeser distribusi masukan; keterbatasan ini dicatat (Hooker et al., 2019).

In [ ]:
import cv2
from scipy.stats import wilcoxon, spearmanr
NP = S // PATCH
def blur_img(x):
    return cv2.blur(x, (16, 16))
def ke_patch(a):
    return a[:NP * PATCH, :NP * PATCH].reshape(NP, PATCH, NP, PATCH).sum(axis=(1, 3))
def kurva(x, urutan, kelas, mode):
    xb = blur_img(x)
    n = len(urutan)
    langkah = np.linspace(0, n, DEL_INS_STEPS + 1).round().astype(int)
    batch = []
    for k in langkah:
        m = np.zeros(NP * NP, bool); m[urutan[:k]] = True
        m = np.repeat(np.repeat(m.reshape(NP, NP), PATCH, 0), PATCH, 1)
        mm = np.zeros(x.shape[:2], bool); mm[:NP * PATCH, :NP * PATCH] = m
        if mode == "deletion":
            z = np.where(mm[..., None], xb, x)
        else:
            z = np.where(mm[..., None], x, xb)
        batch.append(z)
    p = F_PROB(np.stack(batch))[:, kelas]
    return langkah / n, p

trap = getattr(np, "trapezoid", None) or np.trapz
metode = {"SHAP Partition": SV_PART}
if SV_GRAD is not None:
    metode["SHAP Gradient"] = SV_GRAD
fa_rows, KURVA = [], {}
rng = np.random.default_rng(SEED)
for i, r in man.iterrows():
    x, kls = X[i], int(r.pred)
    peta = {nm: ke_patch(v[i, ..., kls]).ravel() for nm, v in metode.items()}
    if GC is not None:
        peta["Grad-CAM"] = ke_patch(GC[i]).ravel()
    for nm, pv in peta.items():
        urut = np.argsort(-pv, kind="stable")
        for mode in ["deletion", "insertion"]:
            fx, p = kurva(x, urut, kls, mode)
            KURVA.setdefault((nm, mode), []).append(p)
            fa_rows.append({"xai_id": r.xai_id, "group": r.group, "true": r.true, "correct": r.correct, "metode": nm, "mode": mode, "auc": trap(p, fx)})
    for mode in ["deletion", "insertion"]:
        aucs, ps = [], []
        for _ in range(N_URUTAN_ACAK):
            fx, p = kurva(x, rng.permutation(NP * NP), kls, mode); aucs.append(trap(p, fx)); ps.append(p)
        KURVA.setdefault(("Random", mode), []).append(np.mean(ps, axis=0))
        fa_rows.append({"xai_id": r.xai_id, "group": r.group, "true": r.true, "correct": r.correct, "metode": "Random", "mode": mode, "auc": float(np.mean(aucs))})
fa = pd.DataFrame(fa_rows)
fa.to_csv(os.path.join(P["C_tab"], "faithfulness.csv"), index=False)

ring = fa.groupby(["metode", "mode"])["auc"].agg(["mean", "std", "median", "count"]).reset_index()
uji = []
pv = fa.pivot_table(index="xai_id", columns=["mode", "metode"], values="auc")
for mode, arah in [("deletion", "lebih kecil lebih baik"), ("insertion", "lebih besar lebih baik")]:
    for a_, b_ in [("SHAP Partition", "Random"), ("SHAP Gradient", "Random"), ("Grad-CAM", "Random"),
                   ("SHAP Partition", "Grad-CAM"), ("SHAP Partition", "SHAP Gradient")]:
        if (mode, a_) in pv.columns and (mode, b_) in pv.columns:
            d = (pv[(mode, a_)] - pv[(mode, b_)]).dropna()
            try:
                p = wilcoxon(d).pvalue if (d != 0).any() else 1.0
            except ValueError:
                p = np.nan
            uji.append({"mode": mode, "pembanding": f"{a_} vs {b_}", "n": len(d), "median_selisih": d.median(), "p_wilcoxon": p, "arah": arah})
uji = pd.DataFrame(uji)
uji["p_holm"] = dk.stats.holm(uji.p_wilcoxon.to_numpy())
uji.to_csv(os.path.join(P["C_tab"], "faithfulness_uji.csv"), index=False)
display(ring.round(4)); display(uji.round(4))

fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
xs = np.linspace(0, 1, DEL_INS_STEPS + 1)
warna = {"SHAP Partition": "#D1495B", "SHAP Gradient": "#E0A030", "Grad-CAM": "#4C9F70", "Random": "#777777"}
for a, mode in zip(ax, ["deletion", "insertion"]):
    for nm in warna:
        if (nm, mode) not in KURVA:
            continue
        arr = np.array(KURVA[(nm, mode)])
        mu = arr.mean(0); se = arr.std(0, ddof=1) / np.sqrt(len(arr)) if len(arr) > 1 else np.zeros_like(mu)
        a.plot(xs, mu, color=warna[nm], label=nm); a.fill_between(xs, mu - 1.96 * se, mu + 1.96 * se, color=warna[nm], alpha=0.15)
    a.set_xlabel(f"Fraction of patches {'removed' if mode == 'deletion' else 'inserted'}"); a.set_ylabel("p(predicted class)")
    a.set_title(mode.capitalize()); a.set_ylim(0, 1); a.legend()
dk.viz.show_and_save(fig, os.path.join(P["C_fig"], "Gambar_faithfulness_deletion_insertion"))

for r in uji.itertuples():
    print(f"{r.mode} - {r.pembanding}: median selisih AUC {fmt(r.median_selisih, 4)} (n = {r.n}), p Wilcoxon {fmt(r.p_wilcoxon, 4)}, "
          f"p Holm {fmt(r.p_holm, 4)} ({r.arah}).")

### C13.2 Stabilitas peta penjelasan

Untuk setiap sampel, citra utuh diberi gangguan kecil (`STABILITAS_GANGGUAN`: *noise* σ = 2, *brightness* ±5%, JPEG Q = 95) lalu dipotong dengan kotak YOLO yang sama, sehingga variasi deteksi tidak ikut memengaruhi uji stabilitas penjelasan CNN. SHAP *Partition* untuk kelas yang semula diprediksi dihitung ulang dan dibandingkan dengan peta awal pada tingkat *patch*:

- korelasi peringkat Spearman;
- Jaccard *patch* teratas (`TOP_FRAKSI` = 10%);
- SSIM peta yang dinormalisasi ke [0, 1].

Perubahan prediksi dan selisih probabilitas dicatat untuk membedakan ketidakstabilan penjelasan dari perubahan keputusan model.

In [ ]:
from skimage.metrics import structural_similarity as ssim
def norm01(a):
    a = a - a.min()
    return a / (a.max() + 1e-12)
st_rows = []
ntop = max(1, int(round(TOP_FRAKSI * NP * NP)))
for (pt, val) in STABILITAS_GANGGUAN:
    Xp = np.concatenate([CR.crops(dk.perturb.apply_perturbation(IMGS[r.sample_id], pt, val, seed_key=r.xai_id), PB[i:i + 1])
                         for i, r in enumerate(man.itertuples())])
    pp = F_PROB(Xp)
    for i, r in man.iterrows():
        kls = int(r.pred)
        v1, _ = shap_partition(Xp[i:i + 1], outputs=(kls,))
        a0, a1 = ke_patch(SV_PART[i, ..., kls]), ke_patch(v1[0, ..., 0])
        t0_, t1_ = set(np.argsort(-a0.ravel())[:ntop]), set(np.argsort(-a1.ravel())[:ntop])
        st_rows.append({"xai_id": r.xai_id, "group": r.group, "gangguan": f"{pt}={val}", "true": r.true, "correct": r.correct,
                        "spearman": spearmanr(a0.ravel(), a1.ravel()).statistic, "jaccard_top": len(t0_ & t1_) / len(t0_ | t1_),
                        "ssim": ssim(norm01(a0), norm01(a1), data_range=1.0),
                        "prediksi_berubah": int(pp[i].argmax() != kls), "selisih_prob": float(pp[i, kls] - prob0[i, kls])})
stab = pd.DataFrame(st_rows)
stab.to_csv(os.path.join(P["C_tab"], "stability.csv"), index=False)
stab_sum = stab.groupby("gangguan").agg(n=("spearman", "size"), spearman_rerata=("spearman", "mean"), spearman_sd=("spearman", "std"),
                                        spearman_median=("spearman", "median"), jaccard_rerata=("jaccard_top", "mean"),
                                        ssim_rerata=("ssim", "mean"), proporsi_prediksi_berubah=("prediksi_berubah", "mean"),
                                        selisih_prob_rerata=("selisih_prob", "mean")).reset_index()
stab_sum.to_csv(os.path.join(P["C_tab"], "stability_ringkas.csv"), index=False)
display(stab_sum.round(3))

fig, ax = plt.subplots(1, 3, figsize=(15, 4.6))
g_ = list(stab_sum.gangguan)
for a, col, lab in zip(ax, ["spearman", "jaccard_top", "ssim"], ["Spearman rank correlation", f"Top-{int(TOP_FRAKSI * 100)}% patch Jaccard", "SSIM"]):
    a.boxplot([stab.loc[stab.gangguan == g, col].dropna() for g in g_], showfliers=False)
    a.set_xticks(range(1, len(g_) + 1)); a.set_xticklabels([s.replace("gaussian_noise", "noise").replace("brightness", "bright.") for s in g_], rotation=20)
    a.set_ylabel(lab); a.set_ylim(-0.1 if col == "spearman" else 0, 1.02)
dk.viz.show_and_save(fig, os.path.join(P["C_fig"], "Gambar_stabilitas_SHAP"))
for r in stab_sum.itertuples():
    print(f"{r.gangguan}: Spearman rerata {fmt(r.spearman_rerata)} (SD {fmt(r.spearman_sd)}), Jaccard top {fmt(r.jaccard_rerata)}, "
          f"SSIM {fmt(r.ssim_rerata)}; prediksi berubah pada {dk.teks.pct(r.proporsi_prediksi_berubah)} sampel (n = {r.n}).")

### C13.3 *Sanity check*: randomisasi parameter model secara bertingkat

Penjelasan yang setia seharusnya berubah ketika parameter model dirusak (Adebayo et al., 2018). Bobot diganti nilai acak normal (simpangan baku sama dengan bobot asli) secara bertingkat dari lapisan atas: (1) *head* klasifikasi, (2) + blok `top_conv`, (3) + blok 7, (4) + blok 6. Pada setiap tahap, SHAP *Partition* dihitung untuk kelas yang semula diprediksi pada `SANITY_N` sampel, lalu dibandingkan dengan peta asli (Spearman |atribusi| tingkat *patch* dan SSIM). Kemiripan yang tetap tinggi setelah randomisasi menandakan peta lebih mencerminkan struktur citra daripada parameter model.

In [ ]:
def tahap_lapisan(model):
    nama = [l.name for l in model.layers]
    grup = [["severity"], ["top_conv", "top_bn"], [n for n in nama if n.startswith("block7")], [n for n in nama if n.startswith("block6")]]
    return [g for g in grup if g]
sanity_rows = []
if MODEL_SET_XAI == "retrain":
    mr = keras.models.clone_model(CR.model); mr.set_weights(CR.model.get_weights())
    rr = np.random.default_rng(SEED)
    sel = man.groupby("true").head(max(1, SANITY_N // 3)).index[:SANITY_N]
    Xs = X[sel]
    kumulatif = []
    for t_i, grp in enumerate(tahap_lapisan(mr), start=1):
        for ln in grp:
            l = mr.get_layer(ln)
            ws = l.get_weights()
            if ws and not isinstance(l, keras.layers.BatchNormalization):
                l.set_weights([rr.normal(0, w.std() + 1e-8, w.shape).astype(w.dtype) if w.ndim > 1 else w for w in ws])
        kumulatif += grp
        fr = dk.models.CnnRunner(mr, scale=CR.scale, class_order=CR.class_order, pad=CR.pad).predict_fn()
        for j, i in enumerate(sel):
            kls = int(man.loc[i, "pred"])
            v, _ = shap_partition(Xs[j:j + 1], outputs=(kls,), max_evals=SANITY_MAX_EVALS, fprob=fr)
            a0, a1 = np.abs(ke_patch(SV_PART[i, ..., kls])), np.abs(ke_patch(v[0, ..., 0]))
            sanity_rows.append({"tahap": t_i, "lapisan_acak": "+".join(g[0] for g in tahap_lapisan(mr)[:t_i]), "xai_id": man.loc[i, "xai_id"],
                                "spearman_abs": spearmanr(a0.ravel(), a1.ravel()).statistic, "ssim": ssim(norm01(a0), norm01(a1), data_range=1.0)})
sanity = pd.DataFrame(sanity_rows)
if len(sanity):
    sanity.to_csv(os.path.join(P["C_tab"], "sanity_check.csv"), index=False)
    ss = sanity.groupby(["tahap", "lapisan_acak"]).agg(spearman_rerata=("spearman_abs", "mean"), spearman_sd=("spearman_abs", "std"),
                                                       ssim_rerata=("ssim", "mean"), n=("ssim", "size")).reset_index()
    display(ss.round(3))
    fig, ax = plt.subplots(figsize=(8, 4.6))
    ax.errorbar(ss.tahap, ss.spearman_rerata, yerr=ss.spearman_sd, fmt="o-", capsize=4, label="Spearman |SHAP|")
    ax.plot(ss.tahap, ss.ssim_rerata, "s--", label="SSIM")
    ax.set_xticks(ss.tahap); ax.set_xticklabels(["head", "+top", "+block7", "+block6"][:len(ss)])
    ax.set_xlabel("Cascading randomization (from top)"); ax.set_ylabel("Similarity to original map"); ax.set_ylim(-0.2, 1.02); ax.legend()
    dk.viz.show_and_save(fig, os.path.join(P["C_fig"], "Gambar_sanity_check"))
    for r in ss.itertuples():
        print(f"Tahap {r.tahap} ({r.lapisan_acak}): Spearman |SHAP| rerata {fmt(r.spearman_rerata)} (SD {fmt(r.spearman_sd)}), SSIM {fmt(r.ssim_rerata)}, n = {r.n}.")
    akhir = ss.iloc[-1]
    print("Kemiripan turun mendekati nol setelah randomisasi, sehingga peta bergantung pada parameter model."
          if akhir.spearman_rerata < 0.3 else "Kemiripan tetap relatif tinggi setelah randomisasi; peta perlu ditafsirkan hati-hati karena sebagian mencerminkan struktur citra.")
else:
    print("Sanity check hanya dijalankan untuk model latih ulang (arsitektur diketahui).")
LOG("C13 selesai.")

## C14. Paket validasi dokter dan analisis kesepakatan

### C14.1 Paket anonim

Paket disusun agar penilaian dokter tidak bias oleh keluaran AI:

- **Bagian A (*blinded*)**: citra konteks (kotak YOLO) dan ROI diperbesar, tanpa prediksi AI. Dokter menilai keparahan (Ringan/Sedang/Berat/Tidak dapat dinilai) dan keyakinan (1–5).
- **Bagian B**: ROI, prediksi AI, probabilitas, dan *heatmap* SHAP. Dokter menilai persetujuan terhadap prediksi, relevansi *heatmap* (1–5), dan fokus *heatmap* (pada lesi/sebagian/di luar lesi).

Setiap kasus memakai kode acak (`K001`, ...). Kunci kode ↔ sampel disimpan terpisah di `_kunci_internal/` dan tidak ikut dalam paket. Citra disimpan sebagai PNG tanpa metadata. Lembar penilaian `lembar_penilaian.xlsx` memakai *dropdown* (validasi data) untuk mengurangi salah isi. Pengumpulan penilaian mengikuti protokol etik penelitian; hasil notebook ini tidak boleh disebut validasi klinis sebelum penilaian ahli nyata tersedia.

In [ ]:
from PIL import Image
from openpyxl import Workbook
from openpyxl.worksheet.datavalidation import DataValidation
EXP = P["C_exp"]; PKT = os.path.join(EXP, "paket_validasi_dokter"); KEY = os.path.join(EXP, "_kunci_internal"); TERISI = os.path.join(EXP, "terisi")
for d in [PKT, KEY, TERISI, os.path.join(PKT, "bagian_A"), os.path.join(PKT, "bagian_B")]:
    os.makedirs(d, exist_ok=True)
per_kls = max(1, N_KASUS_DOKTER // 3)
kasus = pd.concat([man[man.true == k].head(per_kls) for k in range(3)]).sample(frac=1.0, random_state=SEED).reset_index().rename(columns={"index": "man_idx"})
kasus["case_id"] = [f"K{i + 1:03d}" for i in range(len(kasus))]
kasus[["case_id", "xai_id", "sample_id", "roi_idx", "true", "pred", "correct"]].to_csv(os.path.join(KEY, "kunci_kasus.csv"), index=False)

def simpan_png(arr, path):
    Image.fromarray(arr.astype(np.uint8)).save(path, format="PNG")
for r in kasus.itertuples():
    i = r.man_idx
    img = IMGS[r.sample_id].copy()
    x1, y1, x2, y2 = [int(round(v)) for v in (r.pb_x1, r.pb_y1, r.pb_x2, r.pb_y2)]
    cv2.rectangle(img, (x1, y1), (x2, y2), (255, 212, 0), 2)
    h, w = img.shape[:2]; sc = min(1.0, 800 / max(h, w))
    simpan_png(cv2.resize(img, (int(w * sc), int(h * sc))), os.path.join(PKT, "bagian_A", f"{r.case_id}_konteks.png"))
    simpan_png(cv2.resize(X[i].astype(np.uint8), (448, 448), interpolation=cv2.INTER_CUBIC), os.path.join(PKT, "bagian_A", f"{r.case_id}_roi.png"))
    fig, ax = plt.subplots(1, 2, figsize=(9, 4.8))
    ax[0].imshow(X[i].astype(np.uint8)); ax[0].axis("off"); ax[0].set_title("ROI")
    ax[1].imshow(dk.viz.overlay_heatmap(X[i].astype(np.uint8), SV_PART[i, ..., r.pred])); ax[1].axis("off")
    ax[1].set_title(f"SHAP for AI prediction: {NAMA_KELAS[r.pred]}")
    fig.suptitle(f"{r.case_id} | AI: {NAMA_KELAS[r.pred]} (p = {[r.any_p0, r.any_p1, r.any_p2][r.pred]:.2f}) | "
                 + ", ".join(f"{NAMA_KELAS[k]} {v:.2f}" for k, v in enumerate([r.any_p0, r.any_p1, r.any_p2])), fontsize=12)
    dk.viz.save_fig(fig, os.path.join(PKT, "bagian_B", f"{r.case_id}_ai_shap")); plt.close(fig)

wb = Workbook()
ws = wb.active; ws.title = "Petunjuk"
for line in ["Lembar penilaian validasi ahli: interpretasi SHAP keparahan dermatitis atopik pada bayi",
             "1. Kerjakan Bagian A lebih dulu tanpa membuka folder bagian_B (penilaian buta terhadap AI).",
             "2. Bagian A: nilai keparahan lesi di dalam kotak kuning (Ringan/Sedang/Berat/Tidak dapat dinilai) dan keyakinan 1-5.",
             "3. Bagian B: buka gambar bagian_B, nilai persetujuan terhadap prediksi AI, relevansi heatmap (1 = tidak relevan, 5 = sangat relevan),",
             "   dan fokus heatmap (merah = area yang menaikkan probabilitas kelas prediksi).",
             "4. Isi kolom penilai_id dengan kode penilai (misalnya D1, D2). Satu berkas per penilai. Jangan ubah case_id.",
             "5. Simpan sebagai penilai_<kode>.xlsx dan kirim kembali ke peneliti.",
             "Citra telah dianonimkan. Alat ini adalah pendukung keputusan, bukan diagnosis mandiri."]:
    ws.append([line])
wa = wb.create_sheet("Bagian_A"); wa.append(["case_id", "penilai_id", "keparahan_ahli", "keyakinan_1_5", "catatan"])
wbb = wb.create_sheet("Bagian_B"); wbb.append(["case_id", "penilai_id", "setuju_prediksi_AI", "relevansi_heatmap_1_5", "fokus_heatmap", "catatan"])
for cid in sorted(kasus.case_id):
    wa.append([cid, "", "", "", ""]); wbb.append([cid, "", "", "", "", ""])
n = len(kasus) + 1
for sh, col, opsi in [(wa, "C", '"Ringan,Sedang,Berat,Tidak dapat dinilai"'), (wa, "D", '"1,2,3,4,5"'),
                      (wbb, "C", '"Ya,Tidak"'), (wbb, "D", '"1,2,3,4,5"'), (wbb, "E", '"Pada lesi,Sebagian,Di luar lesi"')]:
    dv = DataValidation(type="list", formula1=opsi, allow_blank=True); sh.add_data_validation(dv); dv.add(f"{col}2:{col}{n}")
for sh in [wa, wbb]:
    for c in "ABCDEF":
        sh.column_dimensions[c].width = 22
wb.properties.creator = None; wb.properties.lastModifiedBy = None
wb.save(os.path.join(PKT, "lembar_penilaian.xlsx"))
zp = shutil.make_archive(os.path.join(EXP, "paket_validasi_dokter"), "zip", PKT)
print(f"Paket: {len(kasus)} kasus ->", zp)
print("Simpan lembar terisi para dokter ke:", TERISI)

### C14.2 Analisis kesepakatan (berjalan bila lembar terisi sudah ada)

Lembar terisi (`terisi/penilai_*.xlsx`) dibaca dan digabung dengan kunci kasus. Ukuran yang dihitung:

- **Antarpenilai, keparahan**: Cohen κ berbobot kuadratik (Cohen, 1968) untuk dua penilai; Krippendorff α ordinal (Krippendorff, 2004) untuk dua penilai atau lebih (menangani data hilang, misalnya "Tidak dapat dinilai").
- **Ahli vs label referensi dataset** dan **ahli vs prediksi AI**: κ berbobot kuadratik per penilai.
- **Relevansi *heatmap***: median, IQR, proporsi skor ≥ 4, dan α ordinal antarpenilai.
- 95% CI dengan *bootstrap* atas kasus (B = `B_BOOT`).

Krippendorff menyarankan α ≥ 0,800 untuk kesimpulan yang dapat diandalkan dan 0,667 ≤ α < 0,800 untuk kesimpulan tentatif. Bila belum ada lembar terisi, sel hanya mencetak status; tidak ada nilai yang dibuat-buat.

In [ ]:
import krippendorff
from sklearn.metrics import cohen_kappa_score
berkas = sorted(glob.glob(os.path.join(TERISI, "*.xlsx")))
kunci = pd.read_csv(os.path.join(KEY, "kunci_kasus.csv"))
PETA_SEV = {"Ringan": 0, "Sedang": 1, "Berat": 2}
if not berkas:
    print("STATUS: belum ada lembar penilaian ahli di", TERISI)
    print("Validasi dokter (checklist 14) belum dapat diklaim. Kirim paket_validasi_dokter.zip ke minimal dua dokter independen.")
    agreement = pd.DataFrame()
else:
    A, Bb = [], []
    for f in berkas:
        rid = os.path.splitext(os.path.basename(f))[0]
        a = pd.read_excel(f, sheet_name="Bagian_A"); b = pd.read_excel(f, sheet_name="Bagian_B")
        a["penilai_id"] = a["penilai_id"].fillna(rid); b["penilai_id"] = b["penilai_id"].fillna(rid)
        A.append(a); Bb.append(b)
    A = pd.concat(A).merge(kunci, on="case_id"); Bb = pd.concat(Bb).merge(kunci, on="case_id")
    A["sev_ahli"] = A["keparahan_ahli"].map(PETA_SEV)
    long = A.merge(Bb[["case_id", "penilai_id", "setuju_prediksi_AI", "relevansi_heatmap_1_5", "fokus_heatmap"]], on=["case_id", "penilai_id"], how="left")
    long.to_csv(os.path.join(EXP, "expert_validation.csv"), index=False)
    raters = sorted(A.penilai_id.unique())
    piv = A.pivot_table(index="case_id", columns="penilai_id", values="sev_ahli")
    pivh = Bb.pivot_table(index="case_id", columns="penilai_id", values="relevansi_heatmap_1_5")
    rng = np.random.default_rng(SEED)
    def boot_ci(fn, df):
        vals = []
        for _ in range(B_BOOT):
            s = df.iloc[rng.integers(0, len(df), len(df))]
            try:
                vals.append(fn(s))
            except Exception:
                vals.append(np.nan)
        return dk.stats.percentile_ci(vals)
    def alpha_ord(df):
        return krippendorff.alpha(reliability_data=df.T.to_numpy(dtype=float), level_of_measurement="ordinal")
    rows = []
    if len(raters) >= 2:
        e = alpha_ord(piv); lo, hi = boot_ci(alpha_ord, piv)
        rows.append({"ukuran": "Krippendorff alpha ordinal: keparahan antarpenilai", "nilai": e, "ci_lo": lo, "ci_hi": hi, "n_kasus": len(piv)})
        e = alpha_ord(pivh); lo, hi = boot_ci(alpha_ord, pivh)
        rows.append({"ukuran": "Krippendorff alpha ordinal: relevansi heatmap antarpenilai", "nilai": e, "ci_lo": lo, "ci_hi": hi, "n_kasus": len(pivh)})
    if len(raters) == 2:
        pp_ = piv.dropna()
        f = lambda d: cohen_kappa_score(d.iloc[:, 0], d.iloc[:, 1], weights="quadratic")
        e = f(pp_); lo, hi = boot_ci(f, pp_)
        rows.append({"ukuran": "Cohen kappa kuadratik: keparahan antarpenilai", "nilai": e, "ci_lo": lo, "ci_hi": hi, "n_kasus": len(pp_)})
    for rid in raters:
        q = A[(A.penilai_id == rid) & A.sev_ahli.notna()]
        for nm, col in [("label referensi dataset", "true"), ("prediksi AI", "pred")]:
            f = lambda d: cohen_kappa_score(d["sev_ahli"].astype(int), d[col].astype(int), weights="quadratic")
            e = f(q); lo, hi = boot_ci(f, q)
            rows.append({"ukuran": f"Kappa kuadratik {rid} vs {nm}", "nilai": e, "ci_lo": lo, "ci_hi": hi, "n_kasus": len(q)})
    h = Bb["relevansi_heatmap_1_5"].dropna().astype(float)
    rows.append({"ukuran": "Relevansi heatmap: median (IQR)", "nilai": h.median(), "ci_lo": h.quantile(.25), "ci_hi": h.quantile(.75), "n_kasus": len(h)})
    rows.append({"ukuran": "Relevansi heatmap: proporsi skor >= 4", "nilai": (h >= 4).mean(), "ci_lo": np.nan, "ci_hi": np.nan, "n_kasus": len(h)})
    rows.append({"ukuran": "Setuju dengan prediksi AI (proporsi)", "nilai": (Bb.setuju_prediksi_AI == "Ya").mean(), "ci_lo": np.nan, "ci_hi": np.nan, "n_kasus": int(Bb.setuju_prediksi_AI.notna().sum())})
    agreement = pd.DataFrame(rows)
    agreement.to_csv(os.path.join(EXP, "agreement.csv"), index=False)
    display(agreement.round(3))
    for r in agreement.itertuples():
        txt = f"{r.ukuran}: {fmt(r.nilai)}" + (f" (95% CI {fci(r.ci_lo, r.ci_hi)})" if np.isfinite(r.ci_lo) and "median" not in r.ukuran else "")
        if "alpha" in r.ukuran:
            txt += "; memenuhi ambang andal Krippendorff (>= 0,800)." if r.nilai >= 0.8 else "; berada pada rentang tentatif (0,667-0,800)." if r.nilai >= 0.667 else "; di bawah ambang tentatif 0,667."
        print(txt)
LOG("C14 selesai.")

## C15. Integrasi prototipe: unggah → YOLO → ROI → CNN → SHAP

### C15.1 Kelas `DermaPrototype`

Alur prototipe memakai fungsi yang sama dengan evaluasi (`YoloRunner`, `CnnRunner`, crop dengan padding yang sama):

1. dekode berkas (gagal → status `GAGAL_DECODE`), konversi *grayscale*/RGBA ke RGB, dan pengecilan citra sangat besar (sisi terpanjang > 1600 piksel);
2. YOLO pada `CONF_THR`; tanpa ROI → status `TANPA_ROI` dengan saran mengambil ulang foto;
3. CNN pada setiap ROI; keparahan tingkat citra = kelas tertinggi di antara ROI;
4. SHAP *Partition* untuk `N_ROI_SHAP_PROTOTIPE` ROI dengan *confidence* deteksi tertinggi;
5. waktu setiap tahap dicatat.

Keluaran selalu memuat pernyataan bahwa prototipe adalah alat pendukung keputusan (*decision-support*), bukan diagnosis mandiri.

In [ ]:
DISCLAIMER = ("Prototipe ini adalah alat pendukung keputusan (decision-support) untuk tenaga kesehatan, "
              "bukan diagnosis mandiri. Hasil wajib dikonfirmasi pemeriksaan klinis.")

class DermaPrototype:
    def __init__(self, yolo, cnn, conf_thr=CONF_THR, n_shap=N_ROI_SHAP_PROTOTIPE, shap_evals=SHAP_MAX_EVALS_PROTO, max_side=1600):
        self.yolo, self.cnn, self.conf_thr, self.n_shap, self.shap_evals, self.max_side = yolo, cnn, conf_thr, n_shap, shap_evals, max_side
        self.fprob = cnn.predict_fn()
        self.explainer = shap.Explainer(self.fprob, shap.maskers.Image(SHAP_MASKER, (cnn.input_size, cnn.input_size, 3)), output_names=NAMA_KELAS)

    def analyze(self, data, with_shap=True):
        t = {}; t0 = time.perf_counter()
        out = {"status": "OK", "pesan": "", "disclaimer": DISCLAIMER, "rois": [], "keparahan_citra": None, "waktu_ms": t}
        try:
            if isinstance(data, (bytes, bytearray)):
                img = dk.data.decode_rgb_bytes(bytes(data)) if len(data) else None
            elif isinstance(data, str):
                img = dk.data.read_rgb(data)
            else:
                img = np.asarray(data)
                img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB) if img.ndim == 2 else img[..., :3]
        except Exception as e:
            img = None; out["pesan"] = str(e)[:200]
        t["decode"] = (time.perf_counter() - t0) * 1000
        if img is None or img.size == 0 or min(img.shape[:2]) < 8:
            out.update(status="GAGAL_DECODE", pesan=out["pesan"] or "Berkas tidak dapat dibaca sebagai citra.")
            t["total"] = (time.perf_counter() - t0) * 1000
            return out, None
        h, w = img.shape[:2]
        if max(h, w) > self.max_side:
            sc = self.max_side / max(h, w); img = cv2.resize(img, (int(w * sc), int(h * sc)), interpolation=cv2.INTER_AREA)
        t1 = time.perf_counter()
        pr = self.yolo.predict([img], conf=self.conf_thr)[0][0]
        t["yolo"] = (time.perf_counter() - t1) * 1000
        if len(pr) == 0:
            out.update(status="TANPA_ROI", pesan="Tidak ada lesi terdeteksi. Ambil ulang foto dengan pencahayaan cukup dan fokus pada area kulit.")
            t["total"] = (time.perf_counter() - t0) * 1000
            return out, img
        t2 = time.perf_counter()
        crops = self.cnn.crops(img, pr[:, :4]); probs = self.cnn.predict(crops)
        t["cnn"] = (time.perf_counter() - t2) * 1000
        for b, p in zip(pr, probs):
            out["rois"].append({"box": [float(v) for v in b[:4]], "det_conf": float(b[4]), "probs": [float(v) for v in p], "pred": int(p.argmax())})
        out["keparahan_citra"] = NAMA_KELAS_ID[int(probs.argmax(1).max())]
        t3 = time.perf_counter()
        if with_shap:
            top = np.argsort(-pr[:, 4])[:self.n_shap]
            for j in top:
                k = int(probs[j].argmax())
                sv = self.explainer(crops[j:j + 1], max_evals=self.shap_evals, batch_size=SHAP_BATCH, outputs=[k], silent=True)
                out["rois"][j]["shap"] = sv.values[0].sum(axis=2)[..., 0]
                out["rois"][j]["crop"] = crops[j].astype(np.uint8)
        t["shap"] = (time.perf_counter() - t3) * 1000
        t["total"] = (time.perf_counter() - t0) * 1000
        return out, img

    def render(self, out, img):
        if img is None:
            return None, []
        vis = img.copy()
        for r in out["rois"]:
            x1, y1, x2, y2 = [int(v) for v in r["box"]]
            col = tuple(int(dk.viz.CLASS_COLORS[r["pred"]][i:i + 2], 16) for i in (1, 3, 5))
            cv2.rectangle(vis, (x1, y1), (x2, y2), col, 2)
            cv2.putText(vis, f"{NAMA_KELAS[r['pred']]} {max(r['probs']):.2f}", (x1, max(12, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, col, 1)
        gal = [(np.uint8(255 * dk.viz.overlay_heatmap(r["crop"], r["shap"])), f"{NAMA_KELAS[r['pred']]} (p={max(r['probs']):.2f})")
               for r in out["rois"] if "shap" in r]
        return vis, gal

PROTO = DermaPrototype(YR, CR)
contoh = man.iloc[0]
o, im = PROTO.analyze(contoh.path)
vis, gal = PROTO.render(o, im)
fig, ax = plt.subplots(1, 1 + len(gal), figsize=(5 * (1 + len(gal)), 5))
ax = np.atleast_1d(ax)
ax[0].imshow(vis); ax[0].axis("off"); ax[0].set_title(f"Severity (max): {o['keparahan_citra']} | ROI: {len(o['rois'])}")
for a, (g, cap) in zip(ax[1:], gal):
    a.imshow(g); a.axis("off"); a.set_title(cap)
dk.viz.show_and_save(fig, os.path.join(P["C_fig"], "Contoh_keluaran_prototipe"))
print({k: round(v, 1) for k, v in o["waktu_ms"].items()}); print(DISCLAIMER)

### C15.2 Uji prototipe

Kasus uji: citra normal ber-lesi, citra multi-lesi, berkas rusak (*byte* acak), berkas kosong, citra tanpa ROI (kulit polos sintetis), citra *grayscale*, PNG RGBA, citra sangat besar (4000×3000), dan citra sangat kecil (64×64). Untuk setiap kasus dicatat status, kecocokan dengan status yang diharapkan, jumlah ROI, keparahan, pesan galat, dan waktu tiap tahap. Status `TANPA_ROI` pada kulit polos adalah perilaku yang diharapkan; bila YOLO tetap mendeteksi ROI, kasus tercatat gagal dan menjadi temuan. Untuk citra *grayscale* dan citra 64×64, yang diuji adalah penanganan masukan tanpa galat; `OK` maupun `TANPA_ROI` diterima karena informasi warna atau resolusi berkurang.

In [ ]:
test_items = dk.pipeline.build_items(split_df, ver, REPO_DIR, "test")
lesi = [i for i in test_items if len(i["gt_cls"]) > 0]
multi = max(test_items, key=lambda i: len(i["gt_cls"]))
img_m = dk.data.read_rgb(multi["path"])
kasus_uji = [(f"normal_{k + 1}", lesi[k]["path"], "OK") for k in range(min(5, len(lesi)))]
kasus_uji += [("multi_lesi", multi["path"], "OK"),
              ("berkas_rusak", np.random.default_rng(SEED).integers(0, 255, 5000, dtype=np.uint8).tobytes(), "GAGAL_DECODE"),
              ("berkas_kosong", b"", "GAGAL_DECODE"),
              ("tanpa_roi_kulit_polos", np.full((512, 512, 3), (224, 182, 160), np.uint8), "TANPA_ROI"),
              ("grayscale", cv2.cvtColor(img_m, cv2.COLOR_RGB2GRAY), "OK|TANPA_ROI"),
              ("png_rgba", cv2.imencode(".png", cv2.cvtColor(img_m, cv2.COLOR_RGB2BGRA))[1].tobytes(), "OK"),
              ("citra_besar_4000x3000", cv2.resize(img_m, (4000, 3000), interpolation=cv2.INTER_CUBIC), "OK"),
              ("citra_kecil_64x64", cv2.resize(img_m, (64, 64), interpolation=cv2.INTER_AREA), "OK|TANPA_ROI")]
hasil = []
for nama, data, harap in kasus_uji:
    try:
        o, _ = PROTO.analyze(data)
        err = ""
    except Exception as e:
        o, err = {"status": "ERROR", "rois": [], "keparahan_citra": None, "waktu_ms": {}, "pesan": str(e)[:200]}, type(e).__name__
    hasil.append({"kasus": nama, "status": o["status"], "status_harapan": harap, "lulus": o["status"] in harap.split("|"),
                  "n_roi": len(o["rois"]), "keparahan_citra": o["keparahan_citra"], "pesan": o.get("pesan", ""), "error": err,
                  **{f"ms_{k}": round(v, 1) for k, v in o["waktu_ms"].items()}})
test_report = pd.DataFrame(hasil)
test_report.to_csv(os.path.join(P["C_proto"], "test_report.csv"), index=False)
display(test_report)
ok = test_report[test_report.status == "OK"]
print(f"Lulus {int(test_report.lulus.sum())} dari {len(test_report)} kasus. "
      f"Median waktu total kasus OK: {fmt(ok.ms_total.median(), 0)} ms (YOLO {fmt(ok.ms_yolo.median(), 0)} ms, CNN {fmt(ok.ms_cnn.median(), 0)} ms, "
      f"SHAP {fmt(ok.ms_shap.median(), 0)} ms untuk {N_ROI_SHAP_PROTOTIPE} ROI).")
gagal = test_report[~test_report.lulus]
if len(gagal):
    print("Kasus tidak sesuai harapan:", ", ".join(f"{a} ({b})" for a, b in zip(gagal.kasus, gagal.status)))

### C15.3 Berkas aplikasi dan antarmuka Gradio

`prototype/app.py` adalah aplikasi Gradio mandiri yang membaca `config_prototipe.json` (lokasi bobot, ambang, skala, urutan kelas) dan paket `dermakit` yang disalin ke folder yang sama, sehingga dapat dijalankan di luar Colab (misalnya `python app.py` atau Hugging Face Spaces). Bila `LAUNCH_GRADIO = True`, antarmuka juga ditampilkan langsung di Colab.

In [ ]:
proto_cfg = {"yolo_path": m_cfg["yolo_path"], "cnn_path": m_cfg["cnn_path"], "cnn_scale": CR.scale, "cnn_class_order": CR.class_order,
             "cnn_pad": CR.pad, "conf_thr": CONF_THR, "nms_iou": MC["inferensi"]["nms_iou"], "yolo_imgsz": MC["inferensi"]["yolo_imgsz"],
             "max_det": MC["inferensi"]["max_det"], "shap_masker": SHAP_MASKER, "shap_max_evals": SHAP_MAX_EVALS_PROTO,
             "n_shap": N_ROI_SHAP_PROTOTIPE, "kelas": NAMA_KELAS, "kelas_id": NAMA_KELAS_ID, "label_map_status": STATUS_LABEL_MAP,
             "disclaimer": DISCLAIMER}
dk.core.save_json(proto_cfg, os.path.join(P["C_proto"], "config_prototipe.json"))
dst = os.path.join(P["C_proto"], "dermakit")
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(os.path.join(P["kode"], "dermakit"), dst, ignore=shutil.ignore_patterns("__pycache__"))
APP = r'''
import os, json, time
import numpy as np, cv2
import torch, torchvision, ultralytics  # PyTorch lebih dulu daripada TensorFlow
import gradio as gr, shap
import dermakit as dk
CFG = json.load(open(os.path.join(os.path.dirname(os.path.abspath(__file__)), "config_prototipe.json")))
import keras
dk.models.get_tf()
YR = dk.models.YoloRunner(CFG["yolo_path"], imgsz=CFG["yolo_imgsz"], nms_iou=CFG["nms_iou"], max_det=CFG["max_det"])
CR = dk.models.CnnRunner(dk.models.load_keras_any(CFG["cnn_path"])[0], scale=CFG["cnn_scale"], class_order=CFG["cnn_class_order"], pad=CFG["cnn_pad"])
EX = shap.Explainer(CR.predict_fn(), shap.maskers.Image(CFG["shap_masker"], (CR.input_size, CR.input_size, 3)), output_names=CFG["kelas"])

def analisis(path):
    if path is None:
        return None, [], "Unggah citra terlebih dahulu."
    img = dk.data.read_rgb(path)
    if img is None:
        return None, [], "Berkas tidak dapat dibaca sebagai citra."
    pr = YR.predict([img], conf=CFG["conf_thr"])[0][0]
    if len(pr) == 0:
        return img, [], "Tidak ada lesi terdeteksi. Ambil ulang foto. " + CFG["disclaimer"]
    crops = CR.crops(img, pr[:, :4]); probs = CR.predict(crops)
    vis, gal = img.copy(), []
    for b, p in zip(pr, probs):
        x1, y1, x2, y2 = [int(v) for v in b[:4]]
        cv2.rectangle(vis, (x1, y1), (x2, y2), (255, 0, 0), 2)
        cv2.putText(vis, f"{CFG['kelas'][int(p.argmax())]} {p.max():.2f}", (x1, max(12, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 0, 0), 1)
    for j in np.argsort(-pr[:, 4])[:CFG["n_shap"]]:
        k = int(probs[j].argmax())
        sv = EX(crops[j:j + 1], max_evals=CFG["shap_max_evals"], batch_size=50, outputs=[k], silent=True).values[0].sum(axis=2)[..., 0]
        gal.append((np.uint8(255 * dk.viz.overlay_heatmap(crops[j].astype(np.uint8), sv)), f"{CFG['kelas'][k]} p={probs[j].max():.2f}"))
    sev = CFG["kelas_id"][int(probs.argmax(1).max())]
    return vis, gal, f"Keparahan tertinggi: {sev} ({len(pr)} ROI). " + CFG["disclaimer"]

demo = gr.Interface(fn=analisis, inputs=gr.Image(type="filepath", label="Citra kulit"),
                    outputs=[gr.Image(label="Deteksi YOLO + CNN"), gr.Gallery(label="SHAP per ROI"), gr.Textbox(label="Ringkasan")],
                    title="YOLO-DERMA: deteksi dan penilaian keparahan dermatitis atopik (prototipe riset)",
                    description=CFG["disclaimer"])
if __name__ == "__main__":
    demo.launch()
'''
with open(os.path.join(P["C_proto"], "app.py"), "w", encoding="utf-8") as f:
    f.write(APP.lstrip())
with open(os.path.join(P["C_proto"], "requirements.txt"), "w") as f:
    f.write("ultralytics\ntensorflow\nshap\ngradio\nopencv-python-headless\nimagehash\nscikit-image\nscipy\npandas\nmatplotlib\n")
print("Prototipe ditulis ke:", P["C_proto"])

if LAUNCH_GRADIO:
    import gradio as gr
    def ui(path):
        o, im = PROTO.analyze(path)
        vis, gal = PROTO.render(o, im)
        ring = f"Status: {o['status']}. " + (f"Keparahan tertinggi: {o['keparahan_citra']} ({len(o['rois'])} ROI). " if o["status"] == "OK" else o["pesan"]) + " " + DISCLAIMER
        return vis, gal, ring
    gr.Interface(fn=ui, inputs=gr.Image(type="filepath"), outputs=[gr.Image(), gr.Gallery(), gr.Textbox()],
                 title="YOLO-DERMA prototipe", description=DISCLAIMER).launch(share=False, debug=False)
LOG("C15 selesai.")

## Ringkasan *checklist* Tahap C dan unduhan

*Checklist* dibaca dari berkas keluaran. Butir 14 hanya ditandai selesai bila `agreement.csv` dari penilaian dokter nyata sudah ada. Folder `PFR_2026_XAI_SHAP` (tanpa salinan kode prototipe) dikemas menjadi ZIP; paket validasi dokter tersedia terpisah sebagai `paket_validasi_dokter.zip`.

In [ ]:
cek = {"11 Implementasi SHAP": ["shap_values/shap_values_partition.npz", "tables/sample_manifest.csv"],
       "12 Heatmap per kelas": ["figures/Gambar_SHAP_per_kelas.png"],
       "13 Faithfulness dan stabilitas": ["tables/faithfulness.csv", "tables/stability.csv"],
       "14 Validasi dokter (penilaian nyata)": ["expert_validation/agreement.csv"],
       "15 Prototipe dan ekspor artikel 2026": ["prototype/test_report.csv", "prototype/app.py"]}
for k, fs in cek.items():
    print(("[x] " if all(os.path.exists(os.path.join(P["C"], x)) for x in fs) else "[ ] ") + k)
print(f"Status label map: {STATUS_LABEL_MAP}")
with open(os.path.join(P["C"], "README_hasil.md"), "w", encoding="utf-8") as f:
    f.write("# PFR_2026_XAI_SHAP\n\n")
    f.write(f"- Model yang dijelaskan: {MODEL_SET_XAI}\n- Status label map: {STATUS_LABEL_MAP}\n")
    f.write("- SHAP hanya menerangkan CNN pada ROI; deteksi YOLO berada di luar cakupan penjelasan.\n")
    f.write(f"- Validasi dokter: {'tersedia' if os.path.exists(os.path.join(EXP, 'agreement.csv')) else 'belum ada penilaian ahli'}\n\n## Berkas\n\n")
    for root, _, files in os.walk(P["C"]):
        for fn in sorted(files):
            f.write(f"- {os.path.relpath(os.path.join(root, fn), P['C'])}\n")
tmp = os.path.join(WORK, "_zip_C")
if os.path.exists(tmp):
    shutil.rmtree(tmp)
shutil.copytree(P["C"], os.path.join(tmp, "PFR_2026_XAI_SHAP"), ignore=shutil.ignore_patterns("dermakit", "_kunci_internal", "__pycache__"))
zp = shutil.make_archive(os.path.join(WORK, f"{NAMA_PROYEK}_PFR_2026_XAI_SHAP"), "zip", tmp)
shutil.copy2(zp, ROOT)
print("ZIP:", zp, f"({os.path.getsize(zp) / 1e6:.1f} MB)")
if IN_COLAB:
    from google.colab import files
    files.download(zp)